In [ ]:
# 03_train_baselines.ipynb

import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import joblib

# Load combined dataset
text = pd.read_csv("../features/text_features.csv")
audio = pd.read_csv("../features/audio_features.csv")
video = pd.read_csv("../features/video_features.csv")
labels = pd.read_csv("../data/labels.csv")

df = text.merge(audio, on="id", how="left").merge(video, on="id", how="left").merge(labels, on="id", how="left")
df = df.fillna(0)

X = df.drop(columns=["id", "label"])
y = df["label"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

models = {
    "LogisticRegression": LogisticRegression(max_iter=200),
    "SVM": SVC(kernel="linear"),
    "RandomForest": RandomForestClassifier(n_estimators=100),
    "NaiveBayes": GaussianNB()
}

os.makedirs("../models/trained_models", exist_ok=True)
os.makedirs("../models/results", exist_ok=True)

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n✅ {name} Accuracy: {acc:.2f}")
    print(classification_report(y_test, y_pred))

    joblib.dump(model, f"../models/trained_models/{name}.pkl")
    results[name] = acc

pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"]).to_csv("../models/results/accuracy_summary.csv", index=False)
print("\n✅ Baseline training complete! Results saved.")
